In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np
import xarray as xr
import rioxarray

data_dir = Path('../data')

In [2]:
metrics_dir = data_dir / 'outputs/plots/metrics/x1-y1-z1/net_cdf'

def read_plot_metrics(plot_id: str):
    metrics = xr.open_dataset(metrics_dir / f"{plot_id}.nc", decode_coords='all')
    metrics.load()
    metrics.close()
    return metrics

In [3]:
plots = gpd.read_file(data_dir / "outputs/plots/plots.geojson")
plots = plots.set_index('id')
plots

,site,plot_number,site_plot_id,geometry
id,,,,
ULO_212_P1,ULO_212,1,ULO_212_P1,"POLYGON ((460601.61 5263976.018, 460562.172 52..."
ULO_212_P2,ULO_212,2,ULO_212_P2,"POLYGON ((460579.795 5263940.548, 460537.296 5..."
ULO_212_P3,ULO_212,3,ULO_212_P3,"POLYGON ((460553.09 5263899.328, 460511.13 526..."
ULO_212_P4,ULO_212,4,ULO_212_P4,"POLYGON ((460527.43 5263862.676, 460486.481 52..."
ULO_212_P5,ULO_212,5,ULO_212_P5,"POLYGON ((460501.219 5263817.156, 460457.934 5..."
ULY_O_27_P1,ULY_O_27,1,ULY_O_27_P1,"POLYGON ((460773.837 5262517.328, 460748.749 5..."
ULY_O_27_P2,ULY_O_27,2,ULY_O_27_P2,"POLYGON ((460705.443 5262465.791, 460664.537 5..."
ULY_O_27_P3,ULY_O_27,3,ULY_O_27_P3,"POLYGON ((460749.487 5262473.667, 460773.099 5..."
ULY_O_27_P4,ULY_O_27,4,ULY_O_27_P4,"POLYGON ((460862.875 5262471.269, 460840.506 5..."


In [4]:
def create_plot_summary(row: gpd.GeoSeries) -> pd.Series:
    id = row.name
    metrics : xr.Dataset = read_plot_metrics(id)

    mean_metrics_names = [
        "point_density",
        "pulse_density",
        "scan_angle_mean",
        "chm",
        "veg_height_mean",
        "veg_height_median",
        "crr",
        "veg_height_q10",
        "veg_height_q20",
        "veg_height_q30",
        "veg_height_q40",
        "veg_height_q50",
        "veg_height_q60",
        "veg_height_q70",
        "veg_height_q80",
        "veg_height_q90",
        "veg_height_sd",
        'veg_height_cv',
        'veg_height_skew',
        'veg_height_kurt',
        'veg_height_gini',
        'canopy_cover_gt1m',
        'canopy_cover_gt1m_w',
        'fhd',
        'fhd_w',
        'vci',
        'vci_w',
        'shann_capture',
        'shann_capture_w',
        'norm_shann_capture',
        'norm_shann_capture_w'
    ]
    # Skip point and pulse density and scan angle
    sd_metric_names = mean_metrics_names[3:]
    # CV for all the ones that are in height units
    cv_metric_names = [
        "chm",
        "veg_height_mean",
        "veg_height_median",
        "veg_height_q10",
        "veg_height_q20",
        "veg_height_q30",
        "veg_height_q40",
        "veg_height_q50",
        "veg_height_q60",
        "veg_height_q70",
        "veg_height_q80",
        "veg_height_q90",
        "veg_height_sd",
    ]
    
 

    mean_metrics : pd.Series = metrics[mean_metrics_names].mean(dim=['x', 'y']).to_pandas()
    sd_metrics : pd.Series = metrics[sd_metric_names].std(dim=['x', 'y']).to_pandas()
    cv_metrics : pd.Series = (sd_metrics[cv_metric_names] / mean_metrics[cv_metric_names])
    mean_metrics = mean_metrics.add_prefix('mean__')
    sd_metrics = sd_metrics.add_prefix('sd__')
    cv_metrics = cv_metrics.add_prefix('cv__')

    # I only want max of chm
    max_metrics = pd.Series({
        "max__chm": metrics['chm'].max(dim=['x', 'y']).item()
    })

    plot_summary_metrics = pd.concat([mean_metrics, max_metrics, sd_metrics, cv_metrics])
    plot_summary_metrics.name = id

    return plot_summary_metrics

In [5]:
create_plot_summary(plots.iloc[0])

mean__point_density      1510.002899
mean__pulse_density      1510.002899
mean__scan_angle_mean       0.000000
mean__chm                  14.592139
mean__veg_height_mean       6.296885
                            ...     
cv__veg_height_q60          0.530008
cv__veg_height_q70          0.475772
cv__veg_height_q80          0.429467
cv__veg_height_q90          0.404084
cv__veg_height_sd           0.526344
Name: ULO_212_P1, Length: 73, dtype: float64

In [6]:
plot_summaries = plots.apply(create_plot_summary, axis=1)
plot_summaries['site'] = plot_summaries.index.str[0:-3]
plot_summaries

,mean__point_density,mean__pulse_density,mean__scan_angle_mean,mean__chm,mean__veg_height_mean,mean__veg_height_median,mean__crr,mean__veg_height_q10,mean__veg_height_q20,mean__veg_height_q30,...,cv__veg_height_q20,cv__veg_height_q30,cv__veg_height_q40,cv__veg_height_q50,cv__veg_height_q60,cv__veg_height_q70,cv__veg_height_q80,cv__veg_height_q90,cv__veg_height_sd,site
id,,,,,,,,,,,,,,,,,,,,,
ULO_212_P1,1510.002899,1510.002899,0.000000,14.592139,6.296885,6.140884,0.476709,2.679675,3.601480,4.473833,...,0.974360,0.799568,0.686518,0.598905,0.530008,0.475772,0.429467,0.404084,0.526344,ULO_212
ULO_212_P2,1262.834526,1262.834526,0.000000,15.537041,6.792791,6.797391,0.504337,2.834882,3.972786,5.014900,...,0.833883,0.673198,0.568876,0.494733,0.447222,0.402306,0.368837,0.355677,0.512356,ULO_212
ULO_212_P3,1474.459824,1474.459824,0.000000,12.065134,4.773995,4.654046,0.458751,1.682085,2.504144,3.221023,...,0.958162,0.782426,0.690177,0.638512,0.613069,0.557949,0.513043,0.497345,0.608474,ULO_212
ULO_212_P4,1633.319608,1633.319608,0.000000,11.372780,4.308096,4.172733,0.455311,1.977987,2.620848,3.172275,...,1.260960,1.042421,0.897646,0.789556,0.713386,0.666517,0.617114,0.585981,0.688254,ULO_212
ULO_212_P5,2625.394410,2625.394410,0.000000,22.018451,7.641270,7.255045,0.394203,3.109681,4.291537,5.401967,...,1.328044,1.122318,0.971113,0.866156,0.784581,0.722131,0.662531,0.614934,0.622397,ULO_212
ULY_O_27_P1,575.722195,331.226850,19.281090,32.784559,21.171254,23.114530,0.589646,7.875354,14.153346,18.065664,...,1.006280,0.841405,0.762997,0.709695,0.678640,0.645674,0.619133,0.591118,0.635328,ULY_O_27
ULY_O_27_P2,469.000661,301.254461,22.515608,22.320238,12.694000,13.613327,0.490324,3.455013,7.333500,9.841965,...,1.646927,1.403391,1.257958,1.155773,1.085185,1.020955,0.969200,0.912997,0.885912,ULY_O_27
ULY_O_27_P3,567.636545,330.156449,-12.805331,29.655143,17.875369,19.565838,0.536443,5.444208,10.964311,14.560353,...,1.202262,1.003803,0.899301,0.831991,0.784228,0.748343,0.705791,0.664029,0.657490,ULY_O_27
ULY_O_27_P4,477.052415,285.809353,-7.180624,30.440904,19.008453,20.894576,0.568313,6.187662,11.940562,15.582298,...,1.107088,0.920220,0.805735,0.736302,0.687781,0.644475,0.603950,0.562285,0.589084,ULY_O_27


In [7]:
csv_dir = Path('../csvs')
plot_summaries.to_csv(csv_dir / 'plot_all_metrics.csv')

In [8]:
site_summaries = plot_summaries.reset_index().drop(columns='id').groupby('site').mean(numeric_only=True)
site_summaries = site_summaries[~site_summaries.index.str.startswith('AGG')]
# site_summaries
site_summaries.to_csv(csv_dir / "site_all_metrics.csv")